# STAC data prep — Groundwater recharge in Africa (two COGs)

Notebook to migrate two regular 2D grids in `.tif` format to **Cloud Optimized GeoTIFFs (CoG)**.
Same pattern as `17_seabed_litho.ipynb`, but with two static layers (two model variants) that are published as two items of one collection.

- **Case:** 2D rasters, continuous (mm/year), Africa at about 0.1 degrees (888 x 721 pixels), `float32`, 1-2.5 MB each.
- **Format:** one COG per file (no tiling needed, see `DATA_FORMATS.md`).
- **Resampling:** `AVERAGE` for overviews, because recharge is a continuous quantity.
- **Nodata:** the two files use different nodata values (`-9999` and `-3.4e38`). Both COGs are written with `-9999`; only masked pixels change.
- **Metadata template:** `metadata/metadata_Recharge.json` (Pazola et al. 2024, Figshare). The empty fields are filled from the rasters below.

Sources (`P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian`):

| File | Model |
|------|-------|
| `RF_recharge_01.tif` | random forest |
| `RF_RK_recharge_01.tif` | random forest + residual kriging |

**Running in docker:** the `P:` drive is not visible in the container, so run from the host:

```powershell
docker\run_notebook_with_p_drive.ps1 -Notebook 20_groundwater_recharge.ipynb -Variable recharge -InputPattern "RF*_recharge_01.tif"
```

It copies the inputs to a local mirror, executes this notebook, copies `stac\` back to `P:`, verifies the hashes and removes the mirror.

### Import packages

In [1]:
import json
import os
from pathlib import Path

import numpy as np
import rasterio
import rioxarray as rxr

### Define drive paths

`DATA_VIVIAN` and `STAC_DATA_REPO` are set in the docker container; otherwise the Windows host paths are used.

In [2]:
COLLECTION_ID = "groundwater_recharge"

# One entry per source file: the item id equals the file name (without .tif)
VARIANTS = {
    "RF_recharge_01": {
        "MODEL": "RF",
        "TITLE": "Groundwater recharge, random forest (0.1 degree)",
        "DESCRIPTION": "Long term average annual groundwater recharge in Africa (1970-2020) at 0.1 degree, predicted with a random forest model.",
    },
    "RF_RK_recharge_01": {
        "MODEL": "RF_RK",
        "TITLE": "Groundwater recharge, random forest with residual kriging (0.1 degree)",
        "DESCRIPTION": "Long term average annual groundwater recharge in Africa (1970-2020) at 0.1 degree, predicted with a random forest model whose residuals are interpolated with kriging and added back to the prediction.",
    },
}
OUT_NODATA = -9999.0

# Same layout as P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian (DATA_VIVIAN is set in docker,
# where the P: drive is not mounted; a local mirror of that folder is used and the stac/ output is copied back to P:)
data_vivian = Path(os.environ.get("DATA_VIVIAN", r"P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian"))
stac_repo = Path(os.environ.get("STAC_DATA_REPO", r"C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\global-coastal-atlas\STAC\data"))

source_tifs = {name: data_vivian / f"{name}.tif" for name in VARIANTS}
metadata_template_path = stac_repo / "metadata" / "metadata_Recharge.json"

processed_data_dir = data_vivian / "stac" / COLLECTION_ID
cog_dir = processed_data_dir / "cog"
cog_dir.mkdir(parents=True, exist_ok=True)

for p in (*source_tifs.values(), metadata_template_path):
    if not p.is_file():
        raise FileNotFoundError(p)
print("Sources:", *source_tifs.values(), sep="\n  ")
print("Metadata template:", metadata_template_path)
print("Output dir:", processed_data_dir)

Sources:
  P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\RF_recharge_01.tif
  P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\RF_RK_recharge_01.tif
Metadata template: C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\global-coastal-atlas\STAC\data\metadata\metadata_Recharge.json
Output dir: P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\stac\groundwater_recharge


### Read raw data

In [3]:
# mask_and_scale=False keeps the stored values and the original nodata
raw = {}
for name, path in source_tifs.items():
    raw[name] = rxr.open_rasterio(path, mask_and_scale=False)
    with rasterio.open(path) as src:
        print(f"{name}: {src.width} x {src.height} | dtype {src.dtypes[0]} | nodata {src.nodata} | crs {src.crs.to_epsg()}")
        print(f"   bounds {tuple(round(v, 4) for v in src.bounds)} | res {tuple(round(v, 6) for v in src.res)}")
        print(f"   tiled {src.is_tiled} | compression {src.compression} | overviews {src.overviews(1)}")

# Both files must be on the same grid to be published as one collection
ref_name, *other_names = list(raw)
for other in other_names:
    assert raw[ref_name].shape == raw[other].shape, "Different grid shapes"
    assert np.allclose(raw[ref_name].rio.bounds(), raw[other].rio.bounds()), "Different bounds"
raw[ref_name]

RF_recharge_01: 888 x 721 | dtype float32 | nodata -9999.0 | crs 4326
   bounds (-25.375, -34.825, 63.475, 37.325) | res (0.100056, 0.100069)
   tiled False | compression None | overviews []
RF_RK_recharge_01: 888 x 721 | dtype float32 | nodata -3.3999999521443642e+38 | crs 4326
   bounds (-25.375, -34.825, 63.475, 37.325) | res (0.100056, 0.100069)
   tiled False | compression Compression.lzw | overviews []


<xarray.DataArray (band: 1, y: 721, x: 888)> Size: 3MB
[640248 values with dtype=float32]
Coordinates:
  * band         (band) int64 8B 1
  * y            (y) float64 6kB 37.27 37.17 37.07 ... -34.57 -34.67 -34.77
  * x            (x) float64 7kB -25.32 -25.22 -25.12 ... 63.22 63.32 63.42
    spatial_ref  int64 8B 0
Attributes:
    AREA_OR_POINT:  Area
    _FillValue:     -9999.0
    scale_factor:   1.0
    add_offset:     0.0

### Check CF compliancy original file

Not applicable for a plain GeoTIFF (same approach as `11b_Salinity` and `17_seabed_litho`): CRS, nodata and attributes are written explicitly below.

### Make CF compliant alterations (dataset dependent)

- Harmonise the nodata value to `-9999` (a pixel is nodata if it equals the source nodata or is not finite).
- Fill the metadata fields that are empty in the template using the rasters: `SPATIAL_EXTENT`, `SPATIAL_RESOLUTION`, `NODATA`, `DATA_TYPE`, `DIMENSIONS`, `MEDIA_TYPE`, `DATA_MODEL`, `COLLECTION_ID`, `UNITS`, ...
- `ITEMS` holds the per-file title, description and model for the STAC builder.

In [4]:
with open(metadata_template_path, encoding="utf-8") as f:
    metadata = json.load(f)

layers = {}
for name, ds in raw.items():
    da = ds.isel(band=0).drop_vars("band")
    src_nodata = ds.rio.nodata
    invalid = ~np.isfinite(da.values)
    if src_nodata is not None:
        invalid |= np.isclose(da.values, src_nodata)
    da = da.where(~invalid, OUT_NODATA).astype("float32")
    da = da.rio.write_crs(4326).rio.write_nodata(OUT_NODATA)
    da.attrs["_FillValue"] = OUT_NODATA
    da.name = name
    layers[name] = da
    valid = da.values[~invalid]
    print(f"{name}: nodata {src_nodata} -> {OUT_NODATA} | valid pixels {valid.size} of {invalid.size} | "
          f"min {valid.min():.2f} mean {valid.mean():.2f} max {valid.max():.2f}")

# Bounds are pixel edges; the STAC bbox must stay inside the valid EPSG:4326 range
left, bottom, right, top = layers[ref_name].rio.bounds()
bbox = [max(left, -180.0), max(bottom, -90.0), min(right, 180.0), min(top, 90.0)]
res_x, res_y = layers[ref_name].rio.resolution()

# Extra info that the raw template does not contain
metadata.update({
    "TITLE_ABBREVIATION": "GroundwaterRecharge",
    "HISTORY": [
        "Pazola et al. (2024) long term average groundwater recharge maps for Africa, published on Figshare",
        "Only the 0.1 degree maps are used: random forest (RF_recharge_01) and random forest with residual kriging (RF_RK_recharge_01)",
        "Converted to Cloud Optimized GeoTIFFs (DEFLATE, floating point predictor, average overviews), nodata harmonised to -9999",
    ],
    "MEDIA_TYPE": "image/tiff; application=geotiff; profile=cloud-optimized",
    "DATA_MODEL": "raster_cog",
    "DIMENSIONS": ["x", "y"],
    "SPATIAL_EXTENT": [round(v, 6) for v in bbox],
    "TEMPORAL_EXTENT": ["1970-01-01T00:00:00Z", "2020-12-31T23:59:59Z"],
    "KEYWORDS": ["groundwater recharge", "african groundwater resources", "groundwater", "machine learning", "random forest", "residual kriging", "Africa"],
    "TAGS": ["groundwater", "recharge", "africa", "cog"],
    "LONG_NAME": "Long term average annual groundwater recharge",
    "UNITS": "mm/year",
    "COMMENT": (
        "Two 0.1 degree maps are included: 'RF_recharge_01' (random forest) and 'RF_RK_recharge_01' "
        "(random forest with residual kriging). Values are long term average annual recharge in mm per annum, "
        f"valid for 1970-2020 (period average, not a time series). {OUT_NODATA:.0f} is nodata. "
        "The pixel size is about 0.1 degree (0.10006 x 0.10007), slightly different from exactly 0.1."
    ),
    "CRS": "EPSG:4326",
    "ITEM_BBOX_CRS": "EPSG:4326",
    "SPATIAL_RESOLUTION": round(abs(res_x), 3),
    "NODATA": OUT_NODATA,
    "DATA_TYPE": "float32",
    "COLLECTION_ID": COLLECTION_ID,
    "WMS_DATASET": COLLECTION_ID,  # GeoServer workspace used by scripts/20_groundwater_recharge.ipynb
    "ITEMS": VARIANTS,
})

# Add attributes to the rasters (GeoTIFF tags only hold scalars)
for name, da in layers.items():
    for attr_name, attr_val in metadata.items():
        if attr_name == "ITEMS":
            continue
        if attr_name == "PROVIDERS":
            attr_val = json.dumps(attr_val)
        if attr_name == "MEDIA_TYPE":
            attr_val = "IMAGE/TIFF"  # same convention as 11b_Salinity
        if isinstance(attr_val, list):
            attr_val = json.dumps(attr_val)
        da.attrs[attr_name] = attr_val
    da.attrs["TITLE"] = VARIANTS[name]["TITLE"]
    da.attrs["MODEL"] = VARIANTS[name]["MODEL"]
    da.attrs["Conventions"] = "CF-1.8"

print("SPATIAL_EXTENT:", metadata["SPATIAL_EXTENT"])
print("SPATIAL_RESOLUTION:", metadata["SPATIAL_RESOLUTION"])
print("NODATA:", metadata["NODATA"], "| DATA_TYPE:", metadata["DATA_TYPE"])

RF_recharge_01: nodata -9999.0 -> -9999.0 | valid pixels 252509 of 640248 | min 0.75 mean 67.38 max 313.00
RF_RK_recharge_01: nodata -3.3999999521443642e+38 -> -9999.0 | valid pixels 252509 of 640248 | min 0.49 mean 66.86 max 440.21
SPATIAL_EXTENT: [-25.375, -34.825, 63.475, 37.325]
SPATIAL_RESOLUTION: 0.1
NODATA: -9999.0 | DATA_TYPE: float32


### Write data to CoG

One CoG per variant, `DEFLATE` compression with the floating point predictor, **average** overviews because the data are continuous.

In [5]:
out_paths = {}
for name, da in layers.items():
    out_paths[name] = cog_dir / f"{name}.tif"
    da.rio.to_raster(
        out_paths[name],
        driver="COG",
        compress="DEFLATE",
        predictor="FLOATING_POINT",
        dtype="float32",
        overview_resampling="AVERAGE",
        resampling="NEAREST",
    )
    print("Wrote COG:", out_paths[name], f"({out_paths[name].stat().st_size / 1e6:.2f} MB)")

metadata_output_path = processed_data_dir / f"metadata_{COLLECTION_ID}.json"
with open(metadata_output_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4, ensure_ascii=False)
print("Wrote metadata:", metadata_output_path)

Wrote COG: P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\stac\groundwater_recharge\cog\RF_recharge_01.tif (0.89 MB)
Wrote COG: P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\stac\groundwater_recharge\cog\RF_RK_recharge_01.tif (0.90 MB)
Wrote metadata: P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\stac\groundwater_recharge\metadata_groundwater_recharge.json


### Verify the output

In [6]:
for name, out_path in out_paths.items():
    with rasterio.open(out_path) as src:
        assert src.dtypes[0] == "float32"
        assert src.nodata == OUT_NODATA
        assert src.crs.to_epsg() == 4326
        assert src.is_tiled, "COG must be internally tiled"
        assert len(src.overviews(1)) > 0, "COG must have overviews"
        new = src.read(1)
        print(f"{name}: block {src.profile.get('blockxsize')} x {src.profile.get('blockysize')} | overviews {src.overviews(1)} | compression {src.compression}")

    with rasterio.open(source_tifs[name]) as src:
        old = src.read(1)
        old_nodata = src.nodata

    # Valid pixels must be identical to the source; only the nodata value was harmonised
    old_invalid = ~np.isfinite(old) | np.isclose(old, old_nodata)
    new_invalid = new == OUT_NODATA
    assert np.array_equal(old_invalid, new_invalid), "Valid-pixel mask changed"
    assert np.array_equal(old[~old_invalid], new[~new_invalid]), "Pixel values changed during COG conversion"
print("OK: groundwater recharge COGs and metadata are ready for the STAC builder notebook.")

RF_recharge_01: block 512 x 512 | overviews [2] | compression Compression.deflate
RF_RK_recharge_01: block 512 x 512 | overviews [2] | compression Compression.deflate
OK: groundwater recharge COGs and metadata are ready for the STAC builder notebook.
